# Fashion-MNIST - Simple CNN Baseline

## 1. Objectives

This notebook trains and evaluates a reusable Simple CNN baseline on Fashion-MNIST. It uses the shared dataset loader, saved train-validation split and DataLoader factory. The baseline is the only experiment run here.

The training framework records sample-weighted loss, accuracy, macro F1-score and epoch duration, selects the best checkpoint using validation accuracy, validation macro F1-score and earlier epoch order, and restores that checkpoint before reporting results.

## 2. Dataset and Preprocessing

The baseline uses the shared `load_dataset()`, `load_split()` and `build_dataloaders()` interfaces. Images are converted to float32 tensors with shape `[B, 1, 28, 28]` and values in `[0, 1]`. No standardization or augmentation is applied. The DataLoader initialization interval is reported as preprocessing time; image conversion and batch loading are included in inference timing.

In [1]:
import json
import platform
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.dl.simple_cnn import build_simple_cnn
from src.preprocessing.dataset import load_dataset
from src.preprocessing.dl import build_dataloaders
from src.preprocessing.split import load_split
from src.training.dl import (
    TrainingConfig,
    count_trainable_parameters,
    evaluate_model,
    fit_model,
    measure_inference_time,
    predict_model,
    set_random_seed,
)

RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "experiments"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "dl"
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {__import__('torchvision').__version__}")
print(f"CPU: {platform.processor()}")
print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")

Project root: c:\Users\SUB\OneDrive - Horváth & Partner GmbH\Asztal\Github\DLProject_FashionMNIST
Python: 3.12.10
PyTorch: 2.7.1+cpu
Torchvision: 0.22.1+cpu
CPU: Intel64 Family 6 Model 181 Stepping 0, GenuineIntel
Device: cpu
CUDA available: False


In [2]:
train_dataset, test_dataset = load_dataset()
train_indices, val_indices = load_split()

BASELINE_PREPROCESSING = {
    "standardize": False,
    "train_augmentation": None,
}
BATCH_SIZE = 64

preprocessing_start = perf_counter()
loaders = build_dataloaders(
    train_dataset,
    test_dataset,
    train_indices,
    val_indices,
    batch_size=BATCH_SIZE,
    seed=RANDOM_SEED,
    **BASELINE_PREPROCESSING,
)
train_eval_loaders = build_dataloaders(
    train_dataset,
    test_dataset,
    train_indices,
    val_indices,
    batch_size=BATCH_SIZE,
    seed=RANDOM_SEED,
    standardize=False,
    train_augmentation=None,
)
preprocessing_time = perf_counter() - preprocessing_start

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]
train_eval_loader = train_eval_loaders["train"]

assert len(train_loader.dataset) == 50_000
assert len(val_loader.dataset) == 10_000
assert len(test_loader.dataset) == 10_000
sample_inputs, sample_targets = next(iter(train_loader))
assert sample_inputs.shape == (BATCH_SIZE, 1, 28, 28)
assert sample_inputs.dtype == torch.float32
assert sample_targets.dtype == torch.long
assert 0.0 <= sample_inputs.min() and sample_inputs.max() <= 1.0
print(f"Training samples: {len(train_loader.dataset):,}")
print(f"Validation samples: {len(val_loader.dataset):,}")
print(f"Test samples: {len(test_loader.dataset):,}")
print(f"Batch shape: {tuple(sample_inputs.shape)}")
print(f"DataLoader initialization time: {preprocessing_time:.4f}s")

Training samples: 50,000
Validation samples: 10,000
Test samples: 10,000
Batch shape: (64, 1, 28, 28)
DataLoader initialization time: 0.0191s


## 3. Baseline Model

The baseline has two convolution-pooling blocks followed by a 128-unit fully connected layer and a ten-class logit output. Dropout is configurable but disabled for CNN-00. The convolutional layers remain separately accessible for later feature-map visualization.

In [3]:
MODEL_PARAMS = {"dropout": 0.0}
model = build_simple_cnn(**MODEL_PARAMS).to(DEVICE)
n_parameters = count_trainable_parameters(model)
assert n_parameters == 421_642

smoke_model = build_simple_cnn(**MODEL_PARAMS).to(DEVICE)
smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=0.001)
smoke_criterion = nn.CrossEntropyLoss()
smoke_inputs = torch.randn(2, 1, 28, 28, device=DEVICE)
smoke_targets = torch.tensor([1, 2], device=DEVICE)
smoke_loss = smoke_criterion(smoke_model(smoke_inputs), smoke_targets)
smoke_loss.backward()
smoke_optimizer.step()
assert smoke_model(smoke_inputs).shape == (2, 10)

print(model)
print(f"Trainable parameters: {n_parameters:,}")
print(f"Smoke-test loss: {smoke_loss.item():.4f}")

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Trainable parameters: 421,642
Smoke-test loss: 2.3128


## 4. Training and Evaluation Framework

`fit_model()` creates a fresh Adam optimizer, trains with `model.train()`, evaluates with `model.eval()` and `torch.no_grad()`, and stores a deep copy of the best `state_dict`. Training metrics are measured on the non-augmented evaluation loader. CUDA synchronization is applied around timed sections when a CUDA device is available. Validation inference timing reports both full DataLoader-plus-forward time and forward-only time after one warm-up pass.

## 5. Baseline Training and Results

CNN-00 uses Adam, learning rate 0.001, batch size 64, ten epochs, dropout 0, weight decay 0, CrossEntropyLoss and random seed 42. No hyperparameter optimization or official test evaluation is performed.

In [4]:
RUN_ID = "CNN-00"
TRAINING_CONFIG = TrainingConfig(epochs=10, learning_rate=0.001, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()

set_random_seed(RANDOM_SEED)
model = build_simple_cnn(**MODEL_PARAMS).to(DEVICE)
history, best_epoch, best_state, training_time = fit_model(
    model,
    train_loader,
    train_eval_loader,
    val_loader,
    criterion,
    TRAINING_CONFIG,
    DEVICE,
)
history_df = pd.DataFrame(history)
history_df.insert(0, "run_id", RUN_ID)
display(history_df)
print(f"Best epoch: {best_epoch}")
print(f"Total training time: {training_time:.2f}s")

Epoch 01/10: train_loss=0.3428, train_acc=0.8766, val_loss=0.3382, val_acc=0.8791, val_macro_f1=0.8788, duration=28.70s
Epoch 02/10: train_loss=0.2807, train_acc=0.8960, val_loss=0.2871, val_acc=0.8936, val_macro_f1=0.8925, duration=28.23s
Epoch 03/10: train_loss=0.2368, train_acc=0.9141, val_loss=0.2558, val_acc=0.9096, val_macro_f1=0.9102, duration=28.47s
Epoch 04/10: train_loss=0.2126, train_acc=0.9213, val_loss=0.2500, val_acc=0.9109, val_macro_f1=0.9115, duration=45.99s
Epoch 05/10: train_loss=0.1848, train_acc=0.9324, val_loss=0.2275, val_acc=0.9174, val_macro_f1=0.9183, duration=40.31s
Epoch 06/10: train_loss=0.1606, train_acc=0.9410, val_loss=0.2178, val_acc=0.9216, val_macro_f1=0.9217, duration=39.02s
Epoch 07/10: train_loss=0.1834, train_acc=0.9299, val_loss=0.2539, val_acc=0.9083, val_macro_f1=0.9096, duration=39.04s
Epoch 08/10: train_loss=0.1245, train_acc=0.9555, val_loss=0.2082, val_acc=0.9251, val_macro_f1=0.9250, duration=39.09s
Epoch 09/10: train_loss=0.1242, train_ac

,run_id,epoch,train_loss,train_accuracy,train_macro_f1,val_loss,val_accuracy,val_macro_f1,epoch_duration_s
0,CNN-00,1,0.342808,0.87658,0.876181,0.338208,0.8791,0.878830,28.700866
1,CNN-00,2,0.280737,0.89598,0.894796,0.287137,0.8936,0.892455,28.225978
2,CNN-00,3,0.236802,0.91406,0.914560,0.255760,0.9096,0.910240,28.468509
3,CNN-00,4,0.212556,0.92132,0.921863,0.250049,0.9109,0.911498,45.992719
4,CNN-00,5,0.184848,0.93236,0.932881,0.227524,0.9174,0.918301,40.309967
5,CNN-00,6,0.160601,0.94100,0.940871,0.217760,0.9216,0.921653,39.021377
6,CNN-00,7,0.183412,0.92988,0.930601,0.253900,0.9083,0.909593,39.037957
7,CNN-00,8,0.124518,0.95552,0.955407,0.208214,0.9251,0.924989,39.094908
8,CNN-00,9,0.124209,0.95546,0.955547,0.225650,0.9200,0.920246,38.534594
9,CNN-00,10,0.118514,0.95484,0.954506,0.248550,0.9197,0.919069,38.818345


Best epoch: 8
Total training time: 366.22s


In [5]:
best_row = history_df.loc[history_df["epoch"] == best_epoch].iloc[0]
validation_inference_time, validation_forward_time = measure_inference_time(
    model, val_loader, DEVICE, repeats=5
)
best_train_metrics = evaluate_model(model, train_eval_loader, criterion, DEVICE)
best_val_metrics = evaluate_model(model, val_loader, criterion, DEVICE)
accuracy_gap = best_train_metrics["accuracy"] - best_val_metrics["accuracy"]

checkpoint_path = ARTIFACTS_DIR / "simple_cnn_baseline.pt"
checkpoint = {
    "state_dict": {key: value.cpu() for key, value in best_state.items()},
    "model_params": MODEL_PARAMS,
    "preprocessing_params": {
        "batch_size": BATCH_SIZE,
        **BASELINE_PREPROCESSING,
        "seed": RANDOM_SEED,
    },
    "training_params": {
        "optimizer": "Adam",
        "learning_rate": TRAINING_CONFIG.learning_rate,
        "epochs": TRAINING_CONFIG.epochs,
        "weight_decay": TRAINING_CONFIG.weight_decay,
    },
    "best_epoch": best_epoch,
    "n_parameters": n_parameters,
}
torch.save(checkpoint, checkpoint_path)

loaded_checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
loaded_model = build_simple_cnn(**loaded_checkpoint["model_params"]).to(DEVICE)
loaded_model.load_state_dict(loaded_checkpoint["state_dict"])
loaded_predictions = predict_model(loaded_model, val_loader, DEVICE)
original_predictions = predict_model(model, val_loader, DEVICE)
assert np.array_equal(loaded_predictions, original_predictions)
assert loaded_checkpoint["best_epoch"] == best_epoch

history_df.to_csv(EXPERIMENTS_DIR / "simple_cnn_epoch_history.csv", index=False)
experiment_row = {
    "run_id": RUN_ID,
    "description": "Simple CNN baseline",
    "train_accuracy": best_train_metrics["accuracy"],
    "val_accuracy": best_val_metrics["accuracy"],
    "train_macro_f1": best_train_metrics["macro_f1"],
    "val_macro_f1": best_val_metrics["macro_f1"],
    "accuracy_gap": accuracy_gap,
    "preprocessing_time_s": preprocessing_time,
    "training_time_s": training_time,
    "inference_time_s": validation_inference_time,
    "forward_inference_time_s": validation_forward_time,
    "model_size_mib": checkpoint_path.stat().st_size / 1024**2,
    "n_parameters": n_parameters,
    "best_epoch": best_epoch,
    "model_params": json.dumps(MODEL_PARAMS, sort_keys=True),
    "preprocessing_params": json.dumps(checkpoint["preprocessing_params"], sort_keys=True),
}
pd.DataFrame([experiment_row]).to_csv(EXPERIMENTS_DIR / "simple_cnn_experiments.csv", index=False)

display(history_df)
display(pd.DataFrame([experiment_row]).T)
print(f"Checkpoint: {checkpoint_path}")
print(f"Checkpoint size: {experiment_row['model_size_mib']:.4f} MiB")
print("Checkpoint reload and prediction verification passed.")

,run_id,epoch,train_loss,train_accuracy,train_macro_f1,val_loss,val_accuracy,val_macro_f1,epoch_duration_s
0,CNN-00,1,0.342808,0.87658,0.876181,0.338208,0.8791,0.878830,28.700866
1,CNN-00,2,0.280737,0.89598,0.894796,0.287137,0.8936,0.892455,28.225978
2,CNN-00,3,0.236802,0.91406,0.914560,0.255760,0.9096,0.910240,28.468509
3,CNN-00,4,0.212556,0.92132,0.921863,0.250049,0.9109,0.911498,45.992719
4,CNN-00,5,0.184848,0.93236,0.932881,0.227524,0.9174,0.918301,40.309967
5,CNN-00,6,0.160601,0.94100,0.940871,0.217760,0.9216,0.921653,39.021377
6,CNN-00,7,0.183412,0.92988,0.930601,0.253900,0.9083,0.909593,39.037957
7,CNN-00,8,0.124518,0.95552,0.955407,0.208214,0.9251,0.924989,39.094908
8,CNN-00,9,0.124209,0.95546,0.955547,0.225650,0.9200,0.920246,38.534594
9,CNN-00,10,0.118514,0.95484,0.954506,0.248550,0.9197,0.919069,38.818345


,0
run_id,CNN-00
description,Simple CNN baseline
train_accuracy,0.95552
val_accuracy,0.9251
train_macro_f1,0.955407
val_macro_f1,0.924989
accuracy_gap,0.03042
preprocessing_time_s,0.01912
training_time_s,366.216465
inference_time_s,2.074627


Checkpoint: c:\Users\SUB\OneDrive - Horváth & Partner GmbH\Asztal\Github\DLProject_FashionMNIST\artifacts\dl\simple_cnn_baseline.pt
Checkpoint size: 1.6116 MiB
Checkpoint reload and prediction verification passed.


## Baseline Learning Curves

The saved CNN-00 epoch history is used directly; the baseline is not retrained. The best epoch is highlighted using the validation checkpoint-selection rule.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

import src.training.dl as training_dl

training_dl = importlib.reload(training_dl)
evaluate_model = training_dl.evaluate_model
measure_inference_time = training_dl.measure_inference_time
predict_model = training_dl.predict_model

FIGURES_DIR = PROJECT_ROOT / "results" / "figures" / "simple_cnn"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PALETTE = [
    "#DFE9EF",
    "#BDCEDB",
    "#8FA7BD",
    "#5E7891",
    "#32485F",
    "#1E3247",
    "#0F1B2A",
]
history_df = pd.read_csv(EXPERIMENTS_DIR / "simple_cnn_epoch_history.csv")
best_epoch = int(pd.read_csv(EXPERIMENTS_DIR / "simple_cnn_experiments.csv").loc[0, "best_epoch"])
baseline_history = history_df[history_df["run_id"] == "CNN-00"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
curve_specs = [("accuracy", "Accuracy"), ("loss", "Loss"), ("macro_f1", "Macro F1-score")]
for ax, (metric, title) in zip(axes, curve_specs):
    ax.plot(
        baseline_history["epoch"],
        baseline_history[f"train_{metric}"],
        label="Training",
        color=PALETTE[4],
        marker="o",
    )
    ax.plot(
        baseline_history["epoch"],
        baseline_history[f"val_{metric}"],
        label="Validation",
        color=PALETTE[6],
        marker="o",
    )
    ax.axvline(best_epoch, color=PALETTE[3], linestyle="--", label=f"Best epoch ({best_epoch})")
    ax.set(title=title, xlabel="Epoch", ylabel=title)
    ax.legend()
fig.suptitle("CNN-00 Baseline Learning Curves")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_learning_curves.png", bbox_inches="tight")
plt.show()

## Class-Level Performance

The validation classification report, confusion matrix and class-level F1 comparison are computed from the restored CNN-00 checkpoint. The most frequent confusion pairs are compared descriptively with the earlier EDA cosine-similarity analysis; this comparison does not establish a causal relationship.

In [ ]:
baseline_checkpoint = torch.load(
    ARTIFACTS_DIR / "simple_cnn_baseline.pt", map_location=DEVICE, weights_only=False
)
baseline_model = build_simple_cnn(**baseline_checkpoint["model_params"]).to(DEVICE)
baseline_model.load_state_dict(baseline_checkpoint["state_dict"])
baseline_model.eval()

val_labels = torch.cat([labels for _, labels in val_loader]).numpy()
val_predictions = predict_model(baseline_model, val_loader, DEVICE)
class_names = list(train_dataset.classes)
classification_report_df = pd.DataFrame(
    classification_report(
        val_labels,
        val_predictions,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).T
display(classification_report_df.round(4))

confusion = confusion_matrix(val_labels, val_predictions, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap=sns.light_palette(PALETTE[5], as_cmap=True),
    square=True,
    ax=ax,
    xticklabels=class_names,
    yticklabels=class_names,
)
ax.set(title="CNN-00 Validation Confusion Matrix", xlabel="Predicted class", ylabel="Actual class")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_confusion_matrix.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
class_f1 = classification_report_df.loc[class_names, "f1-score"].sort_values()
class_f1.plot.barh(ax=ax, color=PALETTE[4])
ax.set(title="CNN-00 Validation F1-score by Class", xlabel="F1-score", ylabel="Class")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_class_f1.png", bbox_inches="tight")
plt.show()

off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
top_confusions = [
    (int(i), int(j), int(off_diagonal[i, j]))
    for i, j in zip(*np.unravel_index(np.argsort(off_diagonal.ravel())[-5:], off_diagonal.shape))
][::-1]
train_pixels = train_dataset.data.float().reshape(len(train_dataset), -1).numpy()
class_means = np.stack(
    [
        train_pixels[train_dataset.targets.numpy() == label].mean(axis=0)
        for label in range(len(class_names))
    ]
)
cosine = cosine_similarity(class_means)
np.fill_diagonal(cosine, -1)
top_cosine = [
    (int(i), int(j), float(cosine[i, j]))
    for i, j in zip(*np.unravel_index(np.argsort(cosine.ravel())[-5:], cosine.shape))
][::-1]
print(
    "Most frequent validation confusion pairs:",
    [(class_names[i], class_names[j], count) for i, j, count in top_confusions],
)
print(
    "Highest EDA-style class-mean cosine pairs:",
    [(class_names[i], class_names[j], round(score, 4)) for i, j, score in top_cosine],
)

## Convolutional Feature Maps

Feature maps are shown for the first validation sample in the saved split (`val_indices[0]`). The original raw grayscale image is shown alongside eight channels from each convolutional layer after ReLU and before its corresponding pooling operation.

In [ ]:
feature_index = 0
raw_image = train_dataset.data[val_indices[feature_index]].numpy()
feature_input = torch.from_numpy(raw_image).float().div(255).unsqueeze(0).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    conv1_relu = baseline_model.relu(baseline_model.conv1(feature_input))
    conv2_relu = baseline_model.relu(baseline_model.conv2(baseline_model.pool(conv1_relu)))

fig, axes = plt.subplots(3, 9, figsize=(18, 6))
axes[0, 0].imshow(raw_image, cmap="gray")
axes[0, 0].set_title("Original")
for ax in axes[0, 1:]:
    ax.axis("off")
for row, activations, title in [(1, conv1_relu, "Conv1 + ReLU"), (2, conv2_relu, "Conv2 + ReLU")]:
    axes[row, 0].text(0.5, 0.5, title, ha="center", va="center", rotation=90)
    axes[row, 0].axis("off")
    for channel in range(8):
        axes[row, channel + 1].imshow(activations[0, channel].cpu(), cmap="magma")
        axes[row, channel + 1].set_title(f"Ch {channel}")
        axes[row, channel + 1].axis("off")
fig.suptitle("CNN-00 Feature Maps for Validation Sample 0")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_feature_maps.png", bbox_inches="tight")
plt.show()

## Baseline Key Findings

CNN-00 reaches its best validation accuracy and macro F1 at epoch 8 (0.92510 and 0.924989). Later epochs increase training fit without improving validation performance, indicating a widening generalization gap. Class-level errors and the EDA class-mean cosine similarities are reported together as descriptive evidence only; similar pairs do not prove that cosine similarity caused the observed confusions. The feature maps show progressively transformed local patterns while preserving the documented sample and layer order.

## Hyperparameter and Preprocessing Optimization

The following configuration-driven runs change exactly one baseline factor at a time: CNN-01 changes dropout, CNN-02 changes learning rate, CNN-03 enables training-fitted standardization, and CNN-04 enables training-only `RandomAffine(degrees=8, translate=(0.05, 0.05))`. Each run creates fresh DataLoaders, model and optimizer, uses the saved split and seed 42, restores its best validation checkpoint, and measures training metrics on the non-augmented evaluation loader.

In [ ]:
from torchvision.transforms import RandomAffine

EXPERIMENT_SPECS = [
    {
        "run_id": "CNN-01",
        "description": "dropout=0.3",
        "model_params": {"dropout": 0.3},
        "learning_rate": 0.001,
        "standardize": False,
        "augmentation": None,
    },
    {
        "run_id": "CNN-02",
        "description": "learning_rate=0.0003",
        "model_params": {"dropout": 0.0},
        "learning_rate": 0.0003,
        "standardize": False,
        "augmentation": None,
    },
    {
        "run_id": "CNN-03",
        "description": "standardize=True",
        "model_params": {"dropout": 0.0},
        "learning_rate": 0.001,
        "standardize": True,
        "augmentation": None,
    },
    {
        "run_id": "CNN-04",
        "description": "RandomAffine training-only",
        "model_params": {"dropout": 0.0},
        "learning_rate": 0.001,
        "standardize": False,
        "augmentation": RandomAffine(degrees=8, translate=(0.05, 0.05)),
    },
]

history_path = EXPERIMENTS_DIR / "simple_cnn_epoch_history.csv"
experiments_path = EXPERIMENTS_DIR / "simple_cnn_experiments.csv"
history_all = pd.read_csv(history_path)
experiments_all = pd.read_csv(experiments_path)
run_ids = [spec["run_id"] for spec in EXPERIMENT_SPECS]
history_all = history_all[~history_all["run_id"].isin(run_ids)]
experiments_all = experiments_all[~experiments_all["run_id"].isin(run_ids)]

# Correct CNN-00 inference measurements using the restored checkpoint only.
baseline_total_time, baseline_forward_time = measure_inference_time(
    baseline_model, val_loader, DEVICE, repeats=5
)
experiments_all.loc[experiments_all["run_id"] == "CNN-00", "inference_time_s"] = baseline_total_time
experiments_all.loc[experiments_all["run_id"] == "CNN-00", "forward_inference_time_s"] = (
    baseline_forward_time
)

for spec in EXPERIMENT_SPECS:
    set_random_seed(RANDOM_SEED)
    train_augmentation = spec["augmentation"]
    start = perf_counter()
    experiment_loaders = build_dataloaders(
        train_dataset,
        test_dataset,
        train_indices,
        val_indices,
        batch_size=BATCH_SIZE,
        seed=RANDOM_SEED,
        standardize=spec["standardize"],
        train_augmentation=train_augmentation,
    )
    eval_loaders = build_dataloaders(
        train_dataset,
        test_dataset,
        train_indices,
        val_indices,
        batch_size=BATCH_SIZE,
        seed=RANDOM_SEED,
        standardize=spec["standardize"],
        train_augmentation=None,
    )
    experiment_preprocessing_time = perf_counter() - start
    experiment_model = build_simple_cnn(**spec["model_params"]).to(DEVICE)
    experiment_history, experiment_best_epoch, experiment_best_state, experiment_training_time = (
        training_dl.fit_model(
            experiment_model,
            experiment_loaders["train"],
            eval_loaders["train"],
            experiment_loaders["val"],
            nn.CrossEntropyLoss(),
            TrainingConfig(epochs=10, learning_rate=spec["learning_rate"], weight_decay=0.0),
            DEVICE,
        )
    )
    experiment_history_df = pd.DataFrame(experiment_history)
    experiment_history_df.insert(0, "run_id", spec["run_id"])
    history_all = pd.concat([history_all, experiment_history_df], ignore_index=True)
    train_metrics = evaluate_model(
        experiment_model, eval_loaders["train"], nn.CrossEntropyLoss(), DEVICE
    )
    val_metrics = evaluate_model(
        experiment_model, experiment_loaders["val"], nn.CrossEntropyLoss(), DEVICE
    )
    total_inference_time, forward_inference_time = measure_inference_time(
        experiment_model, experiment_loaders["val"], DEVICE, repeats=5
    )
    normalization_params = {}
    if spec["standardize"]:
        normalize = eval_loaders["train"].dataset.transform.transforms[1]
        normalization_params = {"mean": float(normalize.mean[0]), "std": float(normalize.std[0])}
    checkpoint = {
        "state_dict": {key: value.cpu() for key, value in experiment_best_state.items()},
        "model_params": spec["model_params"],
        "preprocessing_params": {
            "batch_size": BATCH_SIZE,
            "seed": RANDOM_SEED,
            "standardize": spec["standardize"],
            "train_augmentation": spec["description"],
            **normalization_params,
        },
        "training_params": {
            "optimizer": "Adam",
            "learning_rate": spec["learning_rate"],
            "epochs": 10,
            "weight_decay": 0.0,
        },
        "best_epoch": experiment_best_epoch,
        "n_parameters": count_trainable_parameters(experiment_model),
    }
    experiment_checkpoint_path = ARTIFACTS_DIR / f"simple_cnn_{spec['run_id'].lower()}.pt"
    torch.save(checkpoint, experiment_checkpoint_path)
    reloaded = torch.load(experiment_checkpoint_path, map_location=DEVICE, weights_only=False)
    reloaded_model = build_simple_cnn(**reloaded["model_params"]).to(DEVICE)
    reloaded_model.load_state_dict(reloaded["state_dict"])
    assert np.array_equal(
        predict_model(reloaded_model, experiment_loaders["val"], DEVICE),
        predict_model(experiment_model, experiment_loaders["val"], DEVICE),
    )
    experiments_all = pd.concat(
        [
            experiments_all,
            pd.DataFrame(
                [
                    {
                        "run_id": spec["run_id"],
                        "description": spec["description"],
                        "train_accuracy": train_metrics["accuracy"],
                        "val_accuracy": val_metrics["accuracy"],
                        "train_macro_f1": train_metrics["macro_f1"],
                        "val_macro_f1": val_metrics["macro_f1"],
                        "accuracy_gap": train_metrics["accuracy"] - val_metrics["accuracy"],
                        "preprocessing_time_s": experiment_preprocessing_time,
                        "training_time_s": experiment_training_time,
                        "inference_time_s": total_inference_time,
                        "forward_inference_time_s": forward_inference_time,
                        "model_size_mib": experiment_checkpoint_path.stat().st_size / 1024**2,
                        "n_parameters": count_trainable_parameters(experiment_model),
                        "best_epoch": experiment_best_epoch,
                        "model_params": json.dumps(spec["model_params"], sort_keys=True),
                        "preprocessing_params": json.dumps(
                            checkpoint["preprocessing_params"], sort_keys=True
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )
    history_all.to_csv(history_path, index=False)
    experiments_all.to_csv(experiments_path, index=False)
    print(f"{spec['run_id']} completed; checkpoint={experiment_checkpoint_path}")

history_all.to_csv(history_path, index=False)
experiments_all.to_csv(experiments_path, index=False)
display(experiments_all.sort_values("run_id"))

## Optimization Comparison

The comparison figures use the same palette and report only CNN-00 through CNN-04. No combined or 20-epoch configuration is run, and no official test-set evaluation is performed.

In [ ]:
comparison = pd.read_csv(experiments_path).sort_values("run_id")
all_history = pd.read_csv(history_path)
run_order = ["CNN-00", "CNN-01", "CNN-02", "CNN-03", "CNN-04"]
comparison["run_id"] = pd.Categorical(comparison["run_id"], run_order, ordered=True)
comparison = comparison.sort_values("run_id")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(
    data=comparison, x="run_id", y="train_accuracy", ax=axes[0], color=PALETTE[3], label="Training"
)
sns.barplot(
    data=comparison,
    x="run_id",
    y="val_accuracy",
    ax=axes[0],
    color=PALETTE[6],
    alpha=0.75,
    label="Validation",
)
sns.barplot(data=comparison, x="run_id", y="val_macro_f1", ax=axes[1], color=PALETTE[4])
axes[0].set(title="Accuracy by Experiment", ylabel="Accuracy")
axes[1].set(title="Validation Macro F1 by Experiment", ylabel="Macro F1")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_experiment_accuracy_f1.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for run_id, group in all_history.groupby("run_id"):
    axes[0].plot(group["epoch"], group["val_accuracy"], marker="o", label=run_id)
    axes[1].plot(group["epoch"], group["val_loss"], marker="o", label=run_id)
axes[0].set(title="Validation Accuracy Learning Curves", xlabel="Epoch", ylabel="Accuracy")
axes[1].set(title="Validation Loss Learning Curves", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_experiment_learning_curves.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=comparison, x="run_id", y="accuracy_gap", ax=axes[0], color=PALETTE[5])
efficiency = comparison.melt(
    id_vars="run_id",
    value_vars=["training_time_s", "inference_time_s", "model_size_mib"],
    var_name="metric",
    value_name="value",
)
sns.barplot(data=efficiency, x="run_id", y="value", hue="metric", ax=axes[1], palette=PALETTE[2:5])
axes[0].set(title="Accuracy Gap by Experiment", ylabel="Train - Validation Accuracy")
axes[1].set(title="Training Time, Inference Time and Model Size", ylabel="Value")
axes[1].legend(title="Metric")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_experiment_efficiency.png", bbox_inches="tight")
plt.show()

display(
    comparison[
        [
            "run_id",
            "best_epoch",
            "train_accuracy",
            "val_accuracy",
            "train_macro_f1",
            "val_macro_f1",
            "accuracy_gap",
            "preprocessing_time_s",
            "training_time_s",
            "inference_time_s",
            "model_size_mib",
        ]
    ]
)

## Final Model Selection and Evaluation

CNN-00 and CNN-03 are evaluated on the official test split using their saved checkpoints and preprocessing configurations. The final model is selected programmatically by validation accuracy, validation macro F1-score, and smaller model size. CNN-05 and CNN-06 are omitted because of the documented CPU resource constraint.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch import nn

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.models.dl.simple_cnn import build_simple_cnn
from src.preprocessing.dataset import load_dataset
from src.preprocessing.dl import build_dataloaders
from src.preprocessing.split import load_split
from src.training.dl import (
    count_trainable_parameters,
    evaluate_model,
    measure_inference_time,
    predict_model,
)

train_dataset, test_dataset = load_dataset()
train_indices, val_indices = load_split()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "experiments"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "dl"

CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
PALETTE = ["#DFE9EF", "#BDCEDB", "#8FA7BD", "#5E7891", "#32485F", "#1E3247", "#0F1B2A"]
FIGURES_DIR = PROJECT_ROOT / "results" / "figures" / "simple_cnn"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
comparison = pd.read_csv(EXPERIMENTS_DIR / "simple_cnn_experiments.csv")
comparison = comparison[comparison["run_id"].isin(["CNN-00", "CNN-03"])].copy()
selected_row = comparison.sort_values(
    ["val_accuracy", "val_macro_f1", "model_size_mib"], ascending=[False, False, True]
).iloc[0]
assert selected_row["run_id"] == "CNN-03"
checkpoint_paths = {
    "CNN-00": ARTIFACTS_DIR / "simple_cnn_baseline.pt",
    "CNN-03": ARTIFACTS_DIR / "simple_cnn_cnn-03.pt",
}
test_rows, report_rows = [], []
test_predictions = {}
for run_id, checkpoint_path in checkpoint_paths.items():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    preprocessing_params = checkpoint["preprocessing_params"]
    test_loaders = build_dataloaders(
        train_dataset,
        test_dataset,
        train_indices,
        val_indices,
        batch_size=preprocessing_params["batch_size"],
        seed=preprocessing_params["seed"],
        standardize=preprocessing_params["standardize"],
        train_augmentation=None,
    )
    if run_id == "CNN-03":
        assert (
            abs(
                test_loaders["test"].dataset.transform.transforms[-1].mean[0]
                - preprocessing_params["mean"]
            )
            < 1e-12
        )
        assert (
            abs(
                test_loaders["test"].dataset.transform.transforms[-1].std[0]
                - preprocessing_params["std"]
            )
            < 1e-12
        )
    test_model = build_simple_cnn(**checkpoint["model_params"]).to(DEVICE)
    test_model.load_state_dict(checkpoint["state_dict"])
    criterion = nn.CrossEntropyLoss()
    metrics = evaluate_model(test_model, test_loaders["test"], criterion, DEVICE)
    predictions = predict_model(test_model, test_loaders["test"], DEVICE)
    targets = torch.cat([targets for _, targets in test_loaders["test"]]).numpy()
    test_predictions[run_id] = (targets, predictions)
    report = classification_report(
        targets, predictions, target_names=CLASS_NAMES, output_dict=True, zero_division=0
    )
    for label, values in report.items():
        if isinstance(values, dict):
            report_rows.append(
                {
                    "run_id": run_id,
                    "class": label,
                    "precision": values["precision"],
                    "recall": values["recall"],
                    "f1_score": values["f1-score"],
                    "support": values["support"],
                }
            )
        else:
            report_rows.append(
                {
                    "run_id": run_id,
                    "class": label,
                    "precision": values,
                    "recall": values,
                    "f1_score": values,
                    "support": len(targets),
                }
            )
    full_time, forward_time = measure_inference_time(
        test_model, test_loaders["test"], DEVICE, repeats=5
    )
    test_rows.append(
        {
            "run_id": run_id,
            "description": "Baseline" if run_id == "CNN-00" else "Standardized optimized model",
            "test_loss": metrics["loss"],
            "test_accuracy": metrics["accuracy"],
            "test_macro_f1": metrics["macro_f1"],
            "inference_time_s": full_time,
            "forward_inference_time_s": forward_time,
            "model_size_mib": checkpoint_path.stat().st_size / 1024**2,
            "n_parameters": count_trainable_parameters(test_model),
            "best_epoch": checkpoint["best_epoch"],
            "checkpoint": str(checkpoint_path.relative_to(PROJECT_ROOT)),
            "preprocessing_params": json.dumps(preprocessing_params, sort_keys=True),
        }
    )
    cm = confusion_matrix(targets, predictions)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
    )
    ax.set(title=f"{run_id} Test Confusion Matrix", xlabel="Predicted", ylabel="Actual")
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / f"simple_cnn_{run_id.lower()}_test_confusion_matrix.png", bbox_inches="tight"
    )
    plt.close(fig)

final_metrics = pd.DataFrame(test_rows)
final_metrics.to_csv(EXPERIMENTS_DIR / "simple_cnn_final_metrics.csv", index=False)
pd.DataFrame(report_rows).to_csv(
    EXPERIMENTS_DIR / "simple_cnn_classification_report.csv", index=False
)
source_checkpoint = checkpoint_paths[selected_row["run_id"]]
final_checkpoint_path = ARTIFACTS_DIR / "simple_cnn.pt"
final_checkpoint = torch.load(source_checkpoint, map_location="cpu", weights_only=False)
final_checkpoint["selected_run_id"] = selected_row["run_id"]
torch.save(final_checkpoint, final_checkpoint_path)
reloaded_final = torch.load(final_checkpoint_path, map_location=DEVICE, weights_only=False)
assert reloaded_final["selected_run_id"] == "CNN-03"
assert (
    reloaded_final["model_params"]
    == torch.load(source_checkpoint, map_location="cpu", weights_only=False)["model_params"]
)

baseline_loaders = build_dataloaders(
    train_dataset,
    test_dataset,
    train_indices,
    val_indices,
    batch_size=64,
    seed=42,
    standardize=False,
    train_augmentation=None,
)
val_sample, _ = next(iter(baseline_loaders["val"]))
val_sample = val_sample[:1].to(DEVICE)
feature_model = build_simple_cnn(**reloaded_final["model_params"]).to(DEVICE)
feature_model.load_state_dict(reloaded_final["state_dict"])
with torch.no_grad():
    feature_1 = feature_model.relu(feature_model.conv1(val_sample))[0].cpu()
    feature_2 = feature_model.relu(
        feature_model.conv2(feature_model.pool(feature_1.unsqueeze(0).to(DEVICE)))
    )[0].cpu()
fig, axes = plt.subplots(2, 9, figsize=(15, 6))
axes[0, 0].imshow(val_sample[0, 0].cpu(), cmap="gray")
axes[0, 0].set_title("Input")
axes[1, 0].imshow(val_sample[0, 0].cpu(), cmap="gray")
axes[1, 0].set_title("Input")
for channel in range(8):
    axes[0, channel + 1].imshow(feature_1[channel], cmap="viridis")
    axes[0, channel + 1].set_title(f"Conv1 {channel + 1}")
    axes[1, channel + 1].imshow(feature_2[channel], cmap="viridis")
    axes[1, channel + 1].set_title(f"Conv2 {channel + 1}")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("CNN-03 ReLU Feature Maps on the Baseline Validation Sample")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "simple_cnn_final_feature_maps.png", bbox_inches="tight")
plt.close(fig)
display(final_metrics)
print(f"Selected model: {selected_row['run_id']}")
print(f"Final checkpoint: {final_checkpoint_path}")

## Model Persistence and Result Export

The final checkpoint is saved as `artifacts/dl/simple_cnn.pt`; the original baseline checkpoint remains `artifacts/dl/simple_cnn_baseline.pt`. Test metrics and class-level results are exported to `simple_cnn_final_metrics.csv` and `simple_cnn_classification_report.csv`.

## Conclusions and Key Findings

CNN-03 is selected from the existing experiments because it has the highest validation accuracy, with macro F1 and model size used as deterministic tie-breakers. CNN-05 and CNN-06 were not run because CPU resource limits made the planned additional training disproportionate. Test-set results are descriptive for the fixed split and seed; they do not change the selection made from validation data.